## Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
import time

## Datasets

In [ ]:
from pathlib import Path

DATA_DIR = Path('/Users/ms/Desktop/article10_data').expanduser()
RAW_DIR = DATA_DIR / 'raw'
processed_dir = DATA_DIR / 'processed'

processed_dir.mkdir(parents=True, exist_ok=True)

fred_file = RAW_DIR / '2018-12.csv'
liu_wu_file = RAW_DIR / 'LW_monthly.xlsx'

if not fred_file.exists():
    raise FileNotFoundError(f'File not found: {fred_file}')

if not liu_wu_file.exists():
    raise FileNotFoundError(f'File not found: {liu_wu_file}')

print('Raw data directory:', RAW_DIR)
print('Processed data directory:', processed_dir)

#### Raw data import

In [ ]:
fred_md_raw = pd.read_csv(fred_file, header=None)

fred_columns = fred_md_raw.iloc[0].tolist()
fred_transform_codes = pd.Series(fred_md_raw.iloc[1].tolist(), index=fred_columns)

fred_md = fred_md_raw.iloc[2:].copy()
fred_md.columns = fred_columns
fred_md = fred_md.reset_index(drop=True)

liu_wu_raw = pd.read_excel(liu_wu_file, header=8)

liu_wu = liu_wu_raw.copy()
liu_wu = liu_wu.rename(columns={liu_wu.columns[0]: 'date'})
liu_wu.columns = [str(col).strip().replace(' ', '_') for col in liu_wu.columns]

#### Yield data preprocessing

In [ ]:
liu_wu['date'] = pd.PeriodIndex(liu_wu['date'].astype(str), freq='M')
liu_wu = liu_wu[(liu_wu['date'] >= pd.Period('1971-08', freq='M')) & (liu_wu['date'] <= pd.Period('2018-12', freq='M'))].copy()
liu_wu = liu_wu.set_index('date')
yield_columns = liu_wu.columns
liu_wu[yield_columns] = liu_wu[yield_columns].apply(pd.to_numeric, errors='coerce')

#### Forward-rate construction

In [ ]:
forward_rates = pd.DataFrame(index=liu_wu.index)
forward_rates['short_rate'] = liu_wu['12_m']
for n in range(2, 11):
    current_yield = liu_wu[f'{12*n}_m']
    previous_yield = liu_wu[f'{12*(n-1)}_m']
    forward_rates[f'fwd_{n}y'] = n * current_yield - (n - 1) * previous_yield

#### 12-month excess-return construction

In [ ]:
target_maturities = [2, 3, 4, 5, 7, 10]
excess_returns = pd.DataFrame(index=liu_wu.index)

for n in target_maturities:
    current_yield = liu_wu[f'{12*n}_m']
    future_previous_yield = liu_wu[f'{12*(n-1)}_m'].shift(-12)
    short_rate = liu_wu['12_m']
    excess_returns[f'xr_{n}y'] = -(n - 1) * (future_previous_yield - current_yield) + (current_yield - short_rate)

excess_returns = excess_returns.dropna()

#### FRED-MD macro-variable transformations

In [ ]:
fred_md['date'] = pd.PeriodIndex(pd.to_datetime(fred_md['sasdate']), freq='M')
fred_md = fred_md.drop(columns='sasdate').set_index('date')


def transform_series(series, code):
    series = pd.to_numeric(series, errors='coerce')

    if code == 1:
        return series

    elif code == 2:
        return series.diff()

    elif code == 3:
        return series.diff().diff()

    elif code == 4:
        return np.log(series.where(series > 1e-6))

    elif code == 5:
        log_series = np.log(series.where(series > 1e-6))
        return log_series.diff()

    elif code == 6:
        log_series = np.log(series.where(series > 1e-6))
        return log_series.diff().diff()

    elif code == 7:
        percent_change = series.pct_change()
        return percent_change.diff()

    else:
        return series * np.nan


transformed_data = {}

for column in fred_md.columns:
    code = int(float(fred_transform_codes[column]))
    transformed_data[column] = transform_series(fred_md[column], code)

macro_panel = pd.DataFrame(transformed_data, index=fred_md.index)
macro_panel = macro_panel[(macro_panel.index >= pd.Period('1971-08', freq='M')) & (macro_panel.index <= pd.Period('2018-12', freq='M'))]

#### Preliminary merged datasets

In [ ]:
processed_dir.mkdir(parents=True, exist_ok=True)

forward_rates.reset_index().to_csv(processed_dir / 'forward_rates.csv', index=False)
excess_returns.reset_index().to_csv(processed_dir / 'excess_returns.csv', index=False)
macro_panel.reset_index().to_csv(processed_dir / 'macro_panel.csv', index=False)
fred_md.reset_index().to_csv(processed_dir / 'fred_md.csv', index=False)

liu_wu — no target — original Liu–Wu yields

forward_rates — no target — short_rate and fwd_2y–fwd_10y

excess_returns — target variables —
xr_2y, xr_3y, xr_4y, xr_5y, xr_7y, xr_10y

fred_md — no target — original FRED-MD variables

macro_panel — no target — transformed FRED-MD variables

## Model data and OOS setup

In [ ]:
features = forward_rates.columns.tolist()
targets = excess_returns.columns.tolist()

yield_only_data = forward_rates.loc[excess_returns.index].copy()
yield_only_data.index = yield_only_data.index + 12
yield_only_data[targets] = excess_returns.to_numpy()
yield_only_data = yield_only_data.dropna().sort_index()

In [ ]:
yield_only_data.reset_index().to_csv(processed_dir / 'yield_only_data.csv', index=False)

In [ ]:
oos_start = pd.Period('1990-01', freq='M')
oos_dates = yield_only_data.index[yield_only_data.index >= oos_start]

In [ ]:
print('Data:', yield_only_data.index.min(), yield_only_data.index.max(), len(yield_only_data))
print('OOS:', oos_dates.min(), oos_dates.max(), len(oos_dates))
print('First predictor month:', oos_dates.min() - 12)

In [ ]:
macro_features_data = pd.concat([forward_rates, macro_panel], axis=1, join='inner')
macro_features_data = macro_features_data.loc[:, ~macro_features_data.columns.duplicated()]
macro_features_data = macro_features_data.loc[excess_returns.index].copy()

first_predictor_month = oos_start - 12

missing_columns = macro_features_data.columns[macro_features_data.isna().any()].tolist()

late_start_columns = [column for column in missing_columns if macro_features_data.loc[macro_features_data.index <= first_predictor_month, column].notna().sum() == 0]

print('All features with missing values:', missing_columns)
print('Dropping features without history before OOS:', late_start_columns)

macro_features_data = macro_features_data.drop(columns=late_start_columns)
macro_features = macro_features_data.columns.tolist()

macro_features_data.index = macro_features_data.index + 12
macro_features_data[targets] = excess_returns.to_numpy()

macro_data = macro_features_data.dropna().sort_index()
macro_oos_dates = macro_data.index[macro_data.index >= oos_start]

print('Macro+yield data:', macro_data.index.min(), macro_data.index.max(), macro_data.shape)
print('Macro+yield OOS:', macro_oos_dates.min(), macro_oos_dates.max(), len(macro_oos_dates))
print('Number of features:', len(macro_features))
print('History before first OOS date:', (macro_data.index < macro_oos_dates.min()).sum())

In [ ]:
macro_data.reset_index().to_csv(processed_dir / 'macro_data.csv', index=False)

## GBRT model

In [ ]:
def create_gbrt_model(params):
    return GradientBoostingRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        loss='squared_error',
        subsample=1.0,
        min_samples_leaf=1,
        random_state=42
    )


def run_gbrt_for_target(data, target, features, oos_dates, param_grid):
    forecasts = []
    validation_log = []
    start_time = time.perf_counter()

    for i, forecast_date in enumerate(oos_dates, 1):
        history = data[data.index < forecast_date]

        X_history = history[features]
        y_history = history[target]

        split = int(len(history) * 0.85)

        X_train = X_history.iloc[:split]
        y_train = y_history.iloc[:split]

        X_validation = X_history.iloc[split:]
        y_validation = y_history.iloc[split:]

        current_validation = []

        for params in param_grid:
            model = create_gbrt_model(params)
            model.fit(X_train, y_train)

            validation_prediction = model.predict(X_validation)
            validation_mse = mean_squared_error(y_validation, validation_prediction)

            result = {
                'date': forecast_date,
                'target': target,
                **params,
                'validation_mse': validation_mse
            }

            current_validation.append(result)
            validation_log.append(result)

        best_result = min(current_validation, key=lambda x: x['validation_mse'])

        best_params = {'n_estimators': int(best_result['n_estimators']), 'max_depth': int(best_result['max_depth']), 'learning_rate': float(best_result['learning_rate'])}

        final_model = create_gbrt_model(best_params)
        final_model.fit(X_history, y_history)

        X_current = data.loc[[forecast_date], features]
        prediction = final_model.predict(X_current)[0]

        forecasts.append({
            'date': forecast_date,
            'target': target,
            'actual': data.loc[forecast_date, target],
            'prediction': prediction,
            'benchmark': y_history.mean(),
            **best_params
        })

        if i % 12 == 0:
            print(f'{target}: {i}/{len(oos_dates)} months', flush=True)

    forecasts = pd.DataFrame(forecasts).set_index('date')
    validation = pd.DataFrame(validation_log)

    mse_gbrt = (forecasts['actual'] - forecasts['prediction']).pow(2).mean()
    mse_benchmark = (forecasts['actual'] - forecasts['benchmark']).pow(2).mean()
    correlation = forecasts['actual'].corr(forecasts['prediction'])
    r2_oos = 1 - mse_gbrt / mse_benchmark
    elapsed = (time.perf_counter() - start_time) / 60

    metrics = {
        'MSE_GBRT': mse_gbrt,
        'MSE_benchmark': mse_benchmark,
        'correlation': correlation,
        'R2_oos': r2_oos
    }

    print(
        f'\n{target} done: '
        f'R2_oos={r2_oos:.4f}, '
        f'MSE={mse_gbrt:.6f}, '
        f'corr={correlation:.4f}, '
        f'time={elapsed:.1f} min\n',
        flush=True
    )

    return forecasts, validation, metrics

#### yield-only training

In [ ]:
learning_rates = np.logspace(np.log10(0.02), np.log10(0.1), 6)[:-1]
param_grid = [
    {
        'n_estimators': 100,
        'max_depth': 3,
        'learning_rate': rate
    }
    for rate in learning_rates
]

print(learning_rates)

In [ ]:
all_forecasts = {}
all_validation = {}
all_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_gbrt_for_target(data=yield_only_data, target=target, features=features, oos_dates=oos_dates, param_grid=param_grid)

    all_forecasts[target] = forecasts_target
    all_validation[target] = validation_target

    metrics_target['target'] = target
    all_metrics.append(metrics_target)

    metrics_progress = pd.DataFrame(all_metrics).set_index('target')

    print(f'Results for {target}')
    print(" ")

    print('\nR2_oos and correlation by maturity:')
    r2_table = metrics_progress[['R2_oos', 'correlation', 'MSE_GBRT', 'MSE_benchmark']].copy()
    r2_table['R2_oos_percent'] = 100 * r2_table['R2_oos']
    display(r2_table)

    print('\nMost frequently selected parameters:')
    selected_params = forecasts_target[['n_estimators', 'max_depth', 'learning_rate']].value_counts().rename('count').reset_index()
    selected_params['share_percent'] = 100 * selected_params['count'] / len(forecasts_target)
    display(selected_params.head(10))

    print(f'Final correlation {target}: {metrics_target["correlation"]:.4f}')

    pd.to_pickle(all_forecasts, 'gbrt_forecasts_progress.pkl')
    pd.to_pickle(all_validation, 'gbrt_validation_progress.pkl')
    metrics_progress.to_csv('gbrt_metrics_progress.csv')

metrics = pd.DataFrame(all_metrics).set_index('target')

print('\nFinal table across all maturities:')
display(metrics)

#### macro+yield training

In [ ]:
macro_param_grid = [
    {
        'n_estimators': n_estimators,
        'max_depth': 3,
        'learning_rate': learning_rate
    }
    for n_estimators in [50, 100]
    for learning_rate in [0.0175 ,0.0125]
    ]


In [ ]:
macro_forecasts = {}
macro_validation = {}
macro_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_gbrt_for_target(data=macro_data, target=target, features=macro_features, oos_dates=macro_oos_dates, param_grid=macro_param_grid)

    macro_forecasts[target] = forecasts_target
    macro_validation[target] = validation_target

    metrics_target['target'] = target
    macro_metrics.append(metrics_target)

    macro_metrics_progress = pd.DataFrame(macro_metrics).set_index('target')

    print(f'\nMacro+yield: results for {target}\n')

    r2_table = macro_metrics_progress[['R2_oos', 'correlation', 'MSE_GBRT', 'MSE_benchmark']].copy()
    r2_table['R2_oos_percent'] = 100 * r2_table['R2_oos']
    display(r2_table)

    print('\nMost frequently selected parameters:')

    selected_params = forecasts_target[['n_estimators', 'max_depth', 'learning_rate']].value_counts().rename('count').reset_index()
    selected_params['share_percent'] = 100 * selected_params['count'] / len(forecasts_target)
    display(selected_params.head(10))

    print(f'Macro+yield correlation {target}: {metrics_target["correlation"]:.4f}')

    pd.to_pickle(macro_forecasts, 'macro_forecasts_progress.pkl')
    pd.to_pickle(macro_validation, 'macro_validation_progress.pkl')
    macro_metrics_progress.to_csv('macro_metrics_progress.csv')

macro_metrics = pd.DataFrame(macro_metrics).set_index('target')

print('\nFinal macro+yield table:')
display(macro_metrics)

## Random Forest model

In [ ]:
def create_rf_model(params):
    return RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_features=params['max_features'],
        min_samples_leaf=params['min_samples_leaf'],
        min_samples_split=params['min_samples_split'],
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
        criterion='squared_error'
    )

In [ ]:
def run_rf_for_target(data, target, features, oos_dates, param_grid):
    forecasts = []
    validation_log = []
    start_time = time.perf_counter()

    for i, forecast_date in enumerate(oos_dates, 1):
        history = data[data.index < forecast_date]

        X_history = history[features]
        y_history = history[target]

        split = int(len(history) * 0.85)

        X_train = X_history.iloc[:split]
        y_train = y_history.iloc[:split]
        X_validation = X_history.iloc[split:]
        y_validation = y_history.iloc[split:]

        current_validation = []

        for params in param_grid:
            model = create_rf_model(params)
            model.fit(X_train, y_train)

            validation_prediction = model.predict(X_validation)
            validation_mse = mean_squared_error(y_validation, validation_prediction)

            result = {
                'date': forecast_date,
                'target': target,
                **params,
                'validation_mse': validation_mse
            }

            current_validation.append(result)
            validation_log.append(result)

        best_result = min(current_validation, key=lambda x: x['validation_mse'])

        best_params = {
            'n_estimators': int(best_result['n_estimators']),
            'max_depth': best_result['max_depth'],
            'max_features': best_result['max_features'],
            'min_samples_leaf': int(best_result.get('min_samples_leaf', 1)),
            'min_samples_split': int(best_result.get('min_samples_split', 2))
        }

        final_model = create_rf_model(best_params)
        final_model.fit(X_history, y_history)

        X_current = data.loc[[forecast_date], features]
        prediction = final_model.predict(X_current)[0]

        forecasts.append({
            'date': forecast_date,
            'target': target,
            'actual': data.loc[forecast_date, target],
            'prediction': prediction,
            'benchmark': y_history.mean(),
            **best_params
        })

        if i % 12 == 0:
            print(f'{target}: {i}/{len(oos_dates)} months', flush=True)

    forecasts = pd.DataFrame(forecasts).set_index('date')
    validation = pd.DataFrame(validation_log)

    mse_rf = (forecasts['actual'] - forecasts['prediction']).pow(2).mean()
    mse_benchmark = (forecasts['actual'] - forecasts['benchmark']).pow(2).mean()
    correlation = forecasts['actual'].corr(forecasts['prediction'])
    r2_oos = 1 - mse_rf / mse_benchmark

    elapsed = (time.perf_counter() - start_time) / 60

    metrics = {
        'MSE_RF': mse_rf,
        'MSE_benchmark': mse_benchmark,
        'correlation': correlation,
        'R2_oos': r2_oos
    }

    print(f'\n{target} done: R2_oos={r2_oos:.4f}, MSE={mse_rf:.6f}, corr={correlation:.4f}, time={elapsed:.1f} min\n', flush=True)

    return forecasts, validation, metrics

#### yield-only training

In [ ]:
rf_param_grid = [
    {
        'n_estimators': n_estimators,
        'max_depth': None,
        'max_features': max_features,
        'min_samples_leaf': 4,
        'min_samples_split': 2
    }
    for n_estimators in [200, 300]
    for max_features in [0.115, 0.2]
]

In [ ]:
rf_forecasts = {}
rf_validation = {}
rf_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_rf_for_target(data=yield_only_data, target=target, features=features, oos_dates=oos_dates, param_grid=rf_param_grid)

    rf_forecasts[target] = forecasts_target
    rf_validation[target] = validation_target

    metrics_target['target'] = target
    rf_metrics.append(metrics_target)

    rf_metrics_progress = pd.DataFrame(rf_metrics).set_index('target')

    print(f'\nRandom Forest results for {target}:')
    display(rf_metrics_progress)

    print('\nMost frequently selected parameters:')
    selected_params = forecasts_target[['n_estimators', 'max_depth', 'max_features']].value_counts().rename('count').reset_index()
    selected_params['share_percent'] = 100 * selected_params['count'] / len(forecasts_target)
    display(selected_params.head(10))

    pd.to_pickle(rf_forecasts, 'rf_forecasts_progress.pkl')
    pd.to_pickle(rf_validation, 'rf_validation_progress.pkl')
    rf_metrics_progress.to_csv('rf_metrics_progress.csv')

rf_metrics = pd.DataFrame(rf_metrics).set_index('target')
display(rf_metrics)

#### macro+yield training

In [ ]:
rf_ma_param_grid = [
    {
        'n_estimators': 250,
        'max_depth': None,
        'max_features': max_features,
        'min_samples_leaf': 3,
        'min_samples_split': 2
    }
    for max_features in [0.025, 0.04]
]

In [ ]:
rf_macro_forecasts = {}
rf_macro_validation = {}
rf_macro_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_rf_for_target(data=macro_data, target=target, features=macro_features, oos_dates=macro_oos_dates, param_grid=rf_ma_param_grid)

    rf_macro_forecasts[target] = forecasts_target
    rf_macro_validation[target] = validation_target

    metrics_target['target'] = target
    rf_macro_metrics.append(metrics_target)

    rf_macro_metrics_progress = pd.DataFrame(rf_macro_metrics).set_index('target')

    print(f'\nRandom Forest macro+yield results for {target}:')
    display(rf_macro_metrics_progress)

    print('\nMost frequently selected parameters:')
    selected_params = forecasts_target[['n_estimators', 'max_depth', 'max_features']].value_counts().rename('count').reset_index()
    selected_params['share_percent'] = 100 * selected_params['count'] / len(forecasts_target)
    display(selected_params.head(10))

    pd.to_pickle(rf_macro_forecasts, 'rf_macro_forecasts_progress.pkl')
    pd.to_pickle(rf_macro_validation, 'rf_macro_validation_progress.pkl')
    rf_macro_metrics_progress.to_csv('rf_macro_metrics_progress.csv')

rf_macro_metrics = pd.DataFrame(rf_macro_metrics).set_index('target')

print('\nFinal Random Forest macro+yield table:')
display(rf_macro_metrics)

## Extreme Trees model

In [ ]:
def create_extra_trees_model(params):
    return ExtraTreesRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_features=params['max_features'],
        min_samples_leaf=params['min_samples_leaf'],
        min_samples_split=params['min_samples_split'],
        bootstrap=False,
        criterion='squared_error',
        n_jobs=-1,
        random_state=42
    )

In [ ]:
def run_extra_trees_for_target(data, target, features, oos_dates, param_grid):
    forecasts = []
    validation_log = []
    start_time = time.time()

    for i, forecast_date in enumerate(oos_dates, 1):
        history = data.loc[data.index < forecast_date].copy()

        X_history = history[features]
        y_history = history[target]

        split_idx = int(len(history) * 0.85)

        X_train = X_history.iloc[:split_idx]
        y_train = y_history.iloc[:split_idx]
        X_validation = X_history.iloc[split_idx:]
        y_validation = y_history.iloc[split_idx:]

        current_validation = []

        for params in param_grid:
            model = create_extra_trees_model(params)
            model.fit(X_train, y_train)

            validation_prediction = model.predict(X_validation)
            validation_mse = mean_squared_error(y_validation, validation_prediction)

            result = {
                'date': forecast_date,
                'target': target,
                **params,
                'validation_mse': validation_mse
            }

            current_validation.append(result)
            validation_log.append(result)

        best_result = min(current_validation, key=lambda x: x['validation_mse'])

        best_params = {
            'n_estimators': int(best_result['n_estimators']),
            'max_depth': best_result['max_depth'],
            'max_features': best_result['max_features'],
            'min_samples_leaf': int(best_result.get('min_samples_leaf', 3)),
            'min_samples_split': int(best_result.get('min_samples_split', 2)),
        }

        final_model = create_extra_trees_model(best_params)
        final_model.fit(X_history, y_history)

        X_forecast = data.loc[[forecast_date], features]
        prediction = final_model.predict(X_forecast)[0]

        actual = data.loc[forecast_date, target]
        benchmark = y_history.mean()

        forecasts.append({
            'date': forecast_date,
            'target': target,
            'actual': actual,
            'prediction': prediction,
            'benchmark': benchmark,
            **best_params
        })

        if i % 12 == 0 or i == len(oos_dates):
            print(f'{target}: {i}/{len(oos_dates)} ({forecast_date})')

    forecasts = pd.DataFrame(forecasts).set_index('date')
    validation = pd.DataFrame(validation_log)

    mse_extra_trees = (forecasts['actual'] - forecasts['prediction']).pow(2).mean()

    mse_benchmark = (forecasts['actual'] - forecasts['benchmark']).pow(2).mean()

    correlation = forecasts['actual'].corr(forecasts['prediction'])

    r2_oos = 1 - mse_extra_trees / mse_benchmark

    metrics = {
        'MSE_ExtraTrees': mse_extra_trees,
        'MSE_benchmark': mse_benchmark,
        'correlation': correlation,
        'R2_oos': r2_oos
    }

    elapsed_minutes = (time.time() - start_time) / 60

    print(
        f'Extreme Trees {target}: '
        f'R2_oos={r2_oos:.4f}, '
        f'MSE={mse_extra_trees:.6f}, '
        f'corr={correlation:.4f}, '
        f'time={elapsed_minutes:.1f} min'
    )

    return forecasts, validation, metrics

#### yield-only training

In [ ]:
et_param_grid_micro = [
    {
        'n_estimators': 70,
        'max_depth': None,
        'max_features': 'sqrt',
        'min_samples_leaf': leaf,
        'min_samples_split': leaf * 2
    }
    for leaf in [4, 5, 6, 7]
]

In [ ]:
et_all_forecasts = {}
et_all_validation = {}
et_all_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_extra_trees_for_target(data=yield_only_data, target=target, features=features, oos_dates=oos_dates, param_grid=et_param_grid_micro)

    et_all_forecasts[target] = forecasts_target
    et_all_validation[target] = validation_target

    metrics_target['target'] = target
    et_all_metrics.append(metrics_target)

    metrics_progress = pd.DataFrame(et_all_metrics).set_index('target')

    print(f'\nResults for {target}')
    display(metrics_progress)

    print('\nMost frequently selected parameters:')
    selected_params = forecasts_target[['n_estimators', 'max_depth', 'max_features']].value_counts().rename('count').reset_index()

    selected_params['share_percent'] = 100 * selected_params['count'] / len(forecasts_target)

    display(selected_params.head(10))

    pd.to_pickle(et_all_forecasts, 'extra_trees_forecasts_progress.pkl')
    pd.to_pickle(et_all_validation, 'extra_trees_validation_progress.pkl')
    metrics_progress.to_csv('extra_trees_metrics_progress.csv')

extra_trees_metrics = pd.DataFrame(et_all_metrics).set_index('target')

print('\nFinal Extreme Trees table:')
display(extra_trees_metrics)

#### macro+yield training

In [ ]:
def make_et_grid(max_features, leaf=3, split=2):
    return [
        {
            'n_estimators': n_estimators,
            'max_depth': None,
            'max_features': mf,
            'min_samples_leaf': leaf,
            'min_samples_split': split
        }
        for n_estimators in [100, 150]
        for mf in max_features
    ]

In [ ]:
et_param_grid_macro_by_target = {
    'xr_2y':  make_et_grid([0.025, 0.035]),
    'xr_3y':  make_et_grid([0.030, 0.040]),
    'xr_4y':  make_et_grid([0.035, 0.050]),
    'xr_5y':  make_et_grid([0.040, 0.060]),
    'xr_7y':  make_et_grid([0.050, 0.080]),
    'xr_10y': make_et_grid([0.050, 0.080])
}

In [ ]:
macro_et_all_forecasts = {}
macro_et_all_validation = {}
macro_et_all_metrics = []

for target in targets:
    forecasts_target, validation_target, metrics_target = run_extra_trees_for_target(data=macro_data, target=target, features=macro_features, oos_dates=macro_oos_dates, param_grid=et_param_grid_macro_by_target[target])

    macro_et_all_forecasts[target] = forecasts_target
    macro_et_all_validation[target] = validation_target

    metrics_target['target'] = target
    macro_et_all_metrics.append(metrics_target)

    metrics_progress = pd.DataFrame(macro_et_all_metrics).set_index('target')

    print(f'\nMacro Extreme Trees — results for {target}')
    display(metrics_progress)

    pd.to_pickle(macro_et_all_forecasts, 'extra_trees_macro_forecasts_progress.pkl')
    pd.to_pickle(macro_et_all_validation, 'extra_trees_macro_validation_progress.pkl')
    metrics_progress.to_csv('extra_trees_macro_metrics_progress.csv')

macro_extra_trees_metrics = pd.DataFrame(macro_et_all_metrics).set_index('target')

display(macro_extra_trees_metrics)